In [0]:
%sql
CREATE OR REPLACE TABLE workspace.projeto_analytics_2.perfil_risco_modelo_gold
USING DELTA
AS 

SELECT
    an.aeronave_modelo,
    COUNT(an.codigo_ocorrencia2) AS total_eventos,
    SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) AS total_acidentes,
    ROUND(SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) / COUNT(os.codigo_ocorrencia) * 100, 2) AS percentual_acidentes
FROM
    workspace.projeto_analytics_2.aeronautica_aeronave_silver AS an
INNER JOIN
    workspace.projeto_analytics_2.aeronautica_ocorrencia_silver AS os
    ON an.codigo_ocorrencia2 = os.codigo_ocorrencia
GROUP BY 
    an.aeronave_modelo
HAVING 
    COUNT(an.codigo_ocorrencia2) > 5 
ORDER BY
    percentual_acidentes DESC;

In [0]:
%sql
-- SELECT * FROM perfil_risco_modelo 

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.projeto_analytics_2.causa_rais_confiabilidade_motor_gold
USING DELTA
AS

SELECT 
    an.aeronave_motor_tipo,
    an.aeronave_registro_segmento,
    COUNT(an.codigo_ocorrencia2) AS total_eventos,
    SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) AS total_acidentes,
    ROUND(SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) / COUNT(os.codigo_ocorrencia) * 100, 2) AS taxa_gravidade
FROM
    workspace.projeto_analytics_2.aeronautica_aeronave_silver AS an
INNER JOIN
    workspace.projeto_analytics_2.aeronautica_ocorrencia_silver AS os
    ON an.codigo_ocorrencia2 = os.codigo_ocorrencia
GROUP BY
    an.aeronave_motor_tipo,
    an.aeronave_registro_segmento
HAVING
    COUNT(an.codigo_ocorrencia2) > 2   
ORDER BY
    taxa_gravidade DESC,
    total_eventos DESC;

In [0]:
%sql
-- SELECT * FROM causa_rais_confiabilidade_motor

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.projeto_analytics_2.sazonalidade_horario_gold
USING DELTA 
AS 

SELECT 
    os.ocorrencia_uf,
    date_format(os.ocorrencia_dia, "MMMM") AS mes,
    CASE 
        WHEN LEFT(os.ocorrencia_hora, 2) BETWEEN "06" AND "11" THEN "Manha"
        WHEN LEFT(os.ocorrencia_hora, 2) BETWEEN "12" AND "18" THEN "Tarde"
        WHEN LEFT(os.ocorrencia_hora, 2) BETWEEN "19" AND "23" THEN "Noite"
        WHEN os.ocorrencia_hora = "NAO INFORMADO" THEN "NAO INFORMADO" 
        ELSE "MADRUGADA" 
    END AS periodo_dia,
    COUNT(*) AS total_eventos,
    SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) AS total_acidentes
FROM 
    workspace.projeto_analytics_2.aeronautica_ocorrencia_silver AS os
GROUP BY 
    os.ocorrencia_uf,
    mes,
    periodo_dia
ORDER BY
    total_acidentes DESC, 
    total_eventos DESC;

In [0]:
%sql
-- SELECT * from sazonalidade_horario

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.projeto_analytics_2.analise_fase_segmento_gold
USING DELTA 
AS

SELECT 
    an.aeronave_registro_segmento,
    an.aeronave_fase_operacao,
    COUNT(*) AS total_eventos,
    SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) AS total_acidentes,
    ROUND((SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS taxa_risco
FROM
    workspace.projeto_analytics_2.aeronautica_aeronave_silver AS an
INNER JOIN
    workspace.projeto_analytics_2.aeronautica_ocorrencia_silver AS os ON an.codigo_ocorrencia2 = os.codigo_ocorrencia
WHERE 
    an.aeronave_fase_operacao != "***"
    AND an.aeronave_registro_segmento != "***"
GROUP BY
    an.aeronave_registro_segmento,
    an.aeronave_fase_operacao
ORDER BY
    total_acidentes DESC;

In [0]:
%sql
-- SELECT * FROM analise_fase_segmento

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.projeto_analytics_2.analise_causas_raiz_gold
USING DELTA
AS

SELECT
    an.aeronave_registro_segmento,
    f.fator_nome,
    f.fator_aspecto,
    COUNT(os.codigo_ocorrencia) AS total_ocorrencias,
    SUM(CASE WHEN os.ocorrencia_classificacao = "ACIDENTE" THEN 1 ELSE 0 END) AS total_acidentes
FROM
    workspace.projeto_analytics_2.aeronautica_aeronave_silver AS an
INNER JOIN 
    workspace.projeto_analytics_2.aeronautica_ocorrencia_silver AS os ON an.codigo_ocorrencia2 = os.codigo_ocorrencia
INNER JOIN
    workspace.projeto_analytics_2.aeronautica_fator_silver AS f ON os.codigo_ocorrencia = f.codigo_ocorrencia3
WHERE 
    os.ocorrencia_classificacao = "ACIDENTE"
GROUP BY
    an.aeronave_registro_segmento,
    f.fator_nome,
    f.fator_aspecto
ORDER BY
    total_acidentes DESC;


In [0]:
%sql
-- SELECT * FROM analise_causas_raiz